# Predicción del modelo


#1. Carga de modelo

In [1]:
import sys
import hashlib
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

import ipywidgets as widgets
from IPython.display import display, clear_output

# Silenciar warnings de version mismatch en Colab (sklearn 1.6.1 vs 1.5.2 entrenado).
# En producción la API valida la versión y falla
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

#  Config
# Detectar entorno: Colab usa /content/, local usa la estructura del repo
IN_COLAB = "google.colab" in sys.modules
MODEL_PATH = Path("/content/churn_model_v1.joblib") if IN_COLAB else Path("../models/churn_model_v1.joblib")
EXPECTED_SKLEARN_VERSION = "1.5.2"
THRESHOLD = 0.45  # mismo que usa la API en app/main.py

# verificacion de entorno
print(f"Python: {sys.version.split()[0]}")
print(f"sklearn instalado: {sklearn.__version__}")
print(f"sklearn esperado:  {EXPECTED_SKLEARN_VERSION}")
if sklearn.__version__ != EXPECTED_SKLEARN_VERSION:
    print(f"  ⚠️  Mismatch de versión (aceptable solo en este smoke test).")
    print(f"      Para reproducir el entorno exacto en local:")
    print(f"          pip install scikit-learn=={EXPECTED_SKLEARN_VERSION}")

assert MODEL_PATH.exists(), f"No se encontró el modelo en {MODEL_PATH}"

# Hash de integridad
sha256 = hashlib.sha256()
with open(MODEL_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
model_hash = sha256.hexdigest()
print(f"\nModelo: {MODEL_PATH.name}")
print(f"SHA256: {model_hash[:16]}...")

# Carga del pipeline
pipeline = joblib.load(MODEL_PATH)
print(f"\nPipeline cargado: {type(pipeline).__name__}")
print(f"Pasos del pipeline: {[name for name, _ in pipeline.steps]}")

def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Replica src.features.add_derived_features para mantener la celda autocontenida."""
    df = df.copy()
    df["charges_per_month"] = np.where(
        df["tenure_months"] > 0,
        df["total_charges"] / df["tenure_months"],
        df["monthly_charge"],
    )
    df["tickets_per_year"] = np.where(
        df["tenure_months"] > 0,
        df["support_tickets"] / (df["tenure_months"] / 12),
        df["support_tickets"] * 12,
    )
    return df

Python: 3.11.15
sklearn instalado: 1.5.2
sklearn esperado:  1.5.2

Modelo: churn_model_v1.joblib
SHA256: 3848edb7d8afcd3a...

Pipeline cargado: Pipeline
Pasos del pipeline: ['preprocessor', 'classifier']


# 2. Prediccion
### Se edita el dct "mi_cliente" con los valores que se desen y se corre la celda para ver la prediccion

In [ ]:
mi_cliente = {
    "tenure_months": 72,         # 0-72
    "monthly_charge": 75.0,       # 20-120
    "total_charges": 900.0,
    "support_tickets": 3,         # 0-8
    "late_payments": 2,           # 0-6
    "avg_monthly_usage_gb": 100.0,
    "contract_type": "mensual",   # mensual / anual / bianual
    "payment_method": "debito",   # debito / credito / transferencia / efectivo
    "internet_service": "fibra",  # fibra / cable / movil / ninguno
    "has_streaming": 1,           # 0 / 1
    "has_security_pack": 0,       # 0 / 1
    "num_products": 2,            # 1-5
    "region": "centro",           # centro / norte / sur / oeste
    "customer_age": 40,           # 18-78
    "is_promo": 0,                # 0 / 1
}

# Prediccion
X = add_derived(pd.DataFrame([mi_cliente]))
proba = float(pipeline.predict_proba(X)[0, 1])
pred = int(proba >= THRESHOLD)
riesgo = "(!!!)ALTO" if proba >= 0.6 else ("(!)MEDIO" if proba >= 0.3 else "BAJO :)")

# Visualizacion
bar_len = int(proba * 40)
bar = "█" * bar_len + "░" * (40 - bar_len)

print(f"Cliente:")
for k, v in mi_cliente.items():
    print(f"  {k}: {v}")

print(f"\n{'─' * 50}")
print(f"Probabilidad de churn: {proba:.4f}")
print(f"  {bar}  {proba:.1%}")
print(f"  Threshold: {THRESHOLD}")
print(f"  Predicción: {'CHURN' if pred == 1 else 'NO CHURN'}")
print(f"  Riesgo: {riesgo}")
print(f"{'─' * 50}")

# Sensibilidad: cuanto cambia la predicción al variar 1 feature crítica
print(f"\nAnalisis de sensibilidad - moviendo support_tickets:")
for tickets in [0, 2, 4, 6, 8]:
    cliente_var = {**mi_cliente, "support_tickets": tickets}
    X_var = add_derived(pd.DataFrame([cliente_var]))
    p = float(pipeline.predict_proba(X_var)[0, 1])
    bar_v = "█" * int(p * 30)
    print(f"  tickets={tickets}: {p:.4f}  {bar_v}")

Cliente:
  tenure_months: 72
  monthly_charge: 75.0
  total_charges: 900.0
  support_tickets: 3
  late_payments: 2
  avg_monthly_usage_gb: 100.0
  contract_type: mensual
  payment_method: debito
  internet_service: fibra
  has_streaming: 1
  has_security_pack: 0
  num_products: 2
  region: centro
  customer_age: 40
  is_promo: 0

──────────────────────────────────────────────────
Probabilidad de churn: 0.5827
  ███████████████████████░░░░░░░░░░░░░░░░░  58.3%
  Threshold: 0.45
  Predicción: CHURN
  Riesgo: (!)MEDIO
──────────────────────────────────────────────────

Análisis de sensibilidad - moviendo support_tickets:
  tickets=0: 0.4279  ████████████
  tickets=2: 0.5314  ███████████████
  tickets=4: 0.6322  ██████████████████
  tickets=6: 0.7226  █████████████████████
  tickets=8: 0.7979  ███████████████████████
